In [4]:
import csv
import os

class Book:
    def __init__(self, book_id, title, author, genre, total_copies, available_copies=None):
        self.book_id = book_id
        self.title = title
        self.author = author
        self.genre = genre
        self.total_copies = int(total_copies)
        self.available_copies = int(available_copies) if available_copies is not None else int(total_copies)

    def to_dict(self):
        return {
            "book_id": self.book_id,
            "title": self.title,
            "author": self.author,
            "genre": self.genre,
            "total_copies": self.total_copies,
            "available_copies": self.available_copies
        }


class LibrarySystem:
    def __init__(self, data_file="library_data.csv"):
        self.data_file = data_file
        self.books = {}
        self.genres = set()
        self.load_from_csv()
    def load_from_csv(self):
        """Loads book records from a CSV file if it exists."""
        if not os.path.exists(self.data_file):
            return

        try:
            with open(self.data_file, mode="r", newline="", encoding="utf-8") as file:
                reader = csv.DictReader(file)
                for row in reader:
                    book = Book(
                        book_id=row["book_id"],
                        title=row["title"],
                        author=row["author"],
                        genre=row["genre"],
                        total_copies=int(row["total_copies"]),
                        available_copies=int(row["available_copies"])
                    )
                    self.books[book.book_id] = book
                    self.genres.add(book.genre)
        except (IOError, KeyError, ValueError) as e:
            print(f"\n[Warning] Error loading data from file: {e}")

    def save_to_csv(self):
        """Saves current book records to a CSV file."""
        fieldnames = ["book_id", "title", "author", "genre", "total_copies", "available_copies"]
        try:
            with open(self.data_file, mode="w", newline="", encoding="utf-8") as file:
                writer = csv.DictWriter(file, fieldnames=fieldnames)
                writer.writeheader()
                for book in self.books.values():
                    writer.writerow(book.to_dict())
        except IOError as e:
            print(f"\n[Error] Failed to save data to file: {e}")
    def add_book(self):
        print("\n--- Add New Book ---")
        book_id = input("Enter Book ID: ").strip()

        if book_id in self.books:
            print("[Error] A book with this ID already exists.")
            return

        title = input("Enter Title: ").strip()
        author = input("Enter Author: ").strip()
        genre = input("Enter Genre: ").strip().capitalize()

        while True:
            try:
                copies = int(input("Enter Total Copies: "))
                if copies <= 0:
                    print("Copies must be a positive integer. Try again.")
                    continue
                break
            except ValueError:
                print("Invalid input! Please enter a valid number.")

        new_book = Book(book_id, title, author, genre, copies)
        self.books[book_id] = new_book
        self.genres.add(genre)
        self.save_to_csv()
        print(f"\n[Success] '{title}' added successfully!")

    def view_all_books(self):
        print("\n--- All Library Books ---")
        if not self.books:
            print("No books available in the library database.")
            return

        print("-" * 80)
        print(f"{'ID':<8} | {'Title':<25} | {'Author':<18} | {'Genre':<12} | {'Avail/Total':<10}")
        print("-" * 80)
        for book in self.books.values():
            status = f"{book.available_copies}/{book.total_copies}"
            print(f"{book.book_id:<8} | {book.title:<25} | {book.author:<18} | {book.genre:<12} | {status:<10}")
        print("-" * 80)

    def search_books(self):
        print("\n--- Search Books ---")
        print("1. Search by Title/Author")
        print("2. Search by Genre")
        choice = input("Select search option (1-2): ").strip()

        matches = []
        if choice == "1":
            query = input("Enter search term (Title or Author): ").strip().lower()
            matches = [
                b for b in self.books.values()
                if query in b.title.lower() or query in b.author.lower()
            ]
        elif choice == "2":
            if not self.genres:
                print("No genres available.")
                return
            print(f"Available Genres: {', '.join(self.genres)}")
            query = input("Enter Genre: ").strip().lower()
            matches = [b for b in self.books.values() if b.genre.lower() == query]
        else:
            print("[Error] Invalid search option selected.")
            return

        if matches:
            print(f"\nFound {len(matches)} matching book(s):")
            print("-" * 80)
            print(f"{'ID':<8} | {'Title':<25} | {'Author':<18} | {'Genre':<12} | {'Avail/Total':<10}")
            print("-" * 80)
            for book in matches:
                status = f"{book.available_copies}/{book.total_copies}"
                print(f"{book.book_id:<8} | {book.title:<25} | {book.author:<18} | {book.genre:<12} | {status:<10}")
            print("-" * 80)
        else:
            print("No matching books found.")

    def issue_book(self):
        print("\n--- Issue a Book ---")
        book_id = input("Enter Book ID to issue: ").strip()

        if book_id not in self.books:
            print("[Error] Book ID not found.")
            return

        book = self.books[book_id]
        if book.available_copies > 0:
            book.available_copies -= 1
            self.save_to_csv()
            print(f"\n[Success] Book '{book.title}' issued successfully!")
            print(f"Remaining available copies: {book.available_copies}")
        else:
            print(f"\n[Notice] Sorry, all copies of '{book.title}' are currently issued.")

    def return_book(self):
        print("\n--- Return a Book ---")
        book_id = input("Enter Book ID to return: ").strip()

        if book_id not in self.books:
            print("[Error] Book ID not found in library system.")
            return

        book = self.books[book_id]
        if book.available_copies < book.total_copies:
            book.available_copies += 1
            self.save_to_csv()
            print(f"\n[Success] Book '{book.title}' returned successfully!")
            print(f"Current available copies: {book.available_copies}")
        else:
            print("\n[Warning] All copies for this book are already returned.")

    def delete_book(self):
        print("\n--- Delete Book Record ---")
        book_id = input("Enter Book ID to remove: ").strip()

        if book_id in self.books:
            removed_book = self.books.pop(book_id)
            self.save_to_csv()
            print(f"\n[Success] '{removed_book.title}' removed from system.")
        else:
            print("[Error] Book ID not found.")
    def run(self):
        while True:
            print("\n==========================================")
            print("     LIBRARY MANAGEMENT SYSTEM (OOP)      ")
            print("==========================================")
            print("1. Add New Book")
            print("2. View All Books")
            print("3. Search Books (Title/Author/Genre)")
            print("4. Issue a Book")
            print("5. Return a Book")
            print("6. Delete a Book Record")
            print("7. Exit")
            print("==========================================")

            choice = input("Select an option (1-7): ").strip()

            if choice == "1":
                self.add_book()
            elif choice == "2":
                self.view_all_books()
            elif choice == "3":
                self.search_books()
            elif choice == "4":
                self.issue_book()
            elif choice == "5":
                self.return_book()
            elif choice == "6":
                self.delete_book()
            elif choice == "7":
                print("\nSaving data and exiting... Thank you!")
                break
            else:
                print("\n[Error] Invalid choice! Please select a number from 1 to 7.")


if __name__ == "__main__":
    app = LibrarySystem()
    app.run()


     LIBRARY MANAGEMENT SYSTEM (OOP)      
1. Add New Book
2. View All Books
3. Search Books (Title/Author/Genre)
4. Issue a Book
5. Return a Book
6. Delete a Book Record
7. Exit
Select an option (1-7): 2

--- All Library Books ---
No books available in the library database.

     LIBRARY MANAGEMENT SYSTEM (OOP)      
1. Add New Book
2. View All Books
3. Search Books (Title/Author/Genre)
4. Issue a Book
5. Return a Book
6. Delete a Book Record
7. Exit
Select an option (1-7): 101

[Error] Invalid choice! Please select a number from 1 to 7.

     LIBRARY MANAGEMENT SYSTEM (OOP)      
1. Add New Book
2. View All Books
3. Search Books (Title/Author/Genre)
4. Issue a Book
5. Return a Book
6. Delete a Book Record
7. Exit
Select an option (1-7): 7

Saving data and exiting... Thank you!
